In [17]:
import hashlib
import os
import random 
import re
from collections import Counter
from typing import Dict, List, Optional, Tuple

import pandas as pd

In [18]:
class ECGAdvancedConcatenator:
    

    def __init__(self, csv_label_file: Optional[str], data_dir: str, labels: Optional[List[int]] = None) -> None:
        if csv_label_file is not None and not os.path.isfile(csv_label_file):
            raise FileNotFoundError(f"CSV label file not found: {csv_label_file}")
        if not os.path.isdir(data_dir):
            raise FileNotFoundError(f"Data directory not found: {data_dir}")

        self.csv_label_file = csv_label_file
        self.data_dir = data_dir
        self.labels = labels or [0, 1, 2, 3]
        self.label_files: Dict[int, List[str]] = {}
        self.data_cache: Dict[str, pd.DataFrame] = {}

        if self.csv_label_file:
            self._load_label_mapping()
        else:
            self._scan_label_directories()

    def _scan_label_directories(self) -> None:
        for label in self.labels:
            label_dir = os.path.join(self.data_dir, str(label))
            if not os.path.isdir(label_dir):
                raise FileNotFoundError(f"Label directory not found: {label_dir}")
            files = [f for f in os.listdir(label_dir) if f.lower().endswith(".csv")]
            if not files:
                raise FileNotFoundError(f"No CSV files found in {label_dir}")
            self.label_files[label] = sorted(files, key=self._extract_hr_number)

    def _load_label_mapping(self) -> None:
        df = pd.read_csv(self.csv_label_file)
        df.columns = [c.strip() for c in df.columns]
        if "File" not in df.columns or "Label" not in df.columns:
            raise ValueError("CSV must include 'File' and 'Label' columns.")

        for _, row in df.iterrows():
            label = int(row["Label"])
            filename = str(row["File"])
            self.label_files.setdefault(label, []).append(filename)

        for label in self.label_files:
            self.label_files[label] = sorted(self.label_files[label], key=self._extract_hr_number)

    def _get_full_path(self, label: int, filename: str) -> str:
        return os.path.join(self.data_dir, str(label), filename)

    def _load_label_files(self, label: int) -> None:
        if label not in self.label_files:
            raise ValueError(f"Label {label} is not available.")

        for filename in self.label_files[label]:
            full_path = self._get_full_path(label, filename)
            if full_path in self.data_cache:
                continue
            if not os.path.isfile(full_path):
                raise FileNotFoundError(f"File missing for label {label}: {full_path}")
            df = pd.read_csv(full_path)
            df.columns = [c.strip() for c in df.columns]
            self.data_cache[full_path] = df

    @staticmethod
    def _get_duration_from_dataframe(df: pd.DataFrame) -> float:
        if "Time" in df.columns:
            time_series = df["Time"].to_numpy()
            if len(time_series) == 0:
                return 0.0
            return float(time_series[-1] - time_series[0])
        return float(len(df))

    @staticmethod
    def _extract_hr_number(filename: str) -> int:
        base = os.path.splitext(os.path.basename(filename))[0].lower()
        # Support names like hr80.csv and hr80_1.csv by reading the first hr<number> token.
        m = re.search(r"hr(\d+)", base)
        if m:
            return int(m.group(1))

        # Fallback: only parse the first numeric token before '_' to avoid hr80_1 -> 801.
        first_token = base.split("_", 1)[0]
        m2 = re.search(r"(\d+)", first_token)
        if m2:
            return int(m2.group(1))

        raise ValueError(f"Unable to extract HR number from filename: {filename}")

    @staticmethod
    def _offset_time(df: pd.DataFrame, offset: float) -> pd.DataFrame:
        if "Time" not in df.columns:
            return df
        df = df.copy()
        df["Time"] = df["Time"] + offset
        return df


    
    def _build_duration_segment(
        self,
        segments: List[Tuple[int, str]],
        target_seconds: float,
        *,
        rng: Optional[random.Random] = None,
        randomize_truncation: bool = False,
    ) -> Tuple[pd.DataFrame, int, float]:
        label_counts = Counter([label for label, _ in segments])
        majority_label = sorted(label_counts.items(), key=lambda item: (-item[1], item[0]))[0][0]
        majority_ratio = float(label_counts[majority_label] / max(len(segments), 1))

        output_parts = []
        total_duration = 0.0
        time_offset = 0.0

        for label, filename in segments:
            full_path = self._get_full_path(label, filename)
            df = self.data_cache[full_path]
            duration = self._get_duration_from_dataframe(df)
            remaining = target_seconds - total_duration
            if remaining <= 0:
                break
            if duration <= remaining:
                output_parts.append(self._offset_time(df, time_offset))
                total_duration += duration
                if "Time" in df.columns and len(df) > 0:
                    time_offset = output_parts[-1]["Time"].iloc[-1]
                continue

            truncated = df.copy()
            if "Time" in truncated.columns and len(truncated) > 0:
                start_time = float(truncated["Time"].iloc[0])
                end_time = float(truncated["Time"].iloc[-1])
                duration = end_time - start_time
                if randomize_truncation and rng is not None and duration > remaining:
                    max_shift = max(0.0, duration - remaining)
                    window_start = start_time + (rng.random() * max_shift)
                    window_end = window_start + remaining
                    truncated = truncated[(truncated["Time"] >= window_start) & (truncated["Time"] <= window_end)]
                    if len(truncated) == 0:
                        truncated = df[df["Time"] <= window_end].tail(1)
                    truncated = truncated.copy()
                    truncated["Time"] = truncated["Time"] - float(truncated["Time"].iloc[0])
                else:
                    cutoff = start_time + remaining
                    truncated = truncated[truncated["Time"] <= cutoff]
                    if len(truncated) > 0:
                        truncated = truncated.copy()
                        truncated["Time"] = truncated["Time"] - float(truncated["Time"].iloc[0])
            else:
                truncated = truncated.iloc[: int(remaining)]
            output_parts.append(self._offset_time(truncated, time_offset))
            total_duration = target_seconds
            break

        if not output_parts:
            raise ValueError("Unable to build concatenated segment from provided files.")

        return pd.concat(output_parts, ignore_index=True), majority_label, majority_ratio

In [19]:
from pathlib import Path
import os
import random
import sys
from collections import Counter

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from features import extract_features_from_window

# ---- Config ----
RAW_BASE_DIR = str(DATA_DIR / "raw_gen")
OUTPUT_ROOT_DIR = str(DATA_DIR / "concatenated")
DATASET_ID = 7
FEATURES_DIR = str(DATA_DIR / "features" / str(DATASET_ID))

LABELS_FOR_ORDER = [0, 1, 2, 3]
ORDER_REPEATS_PER_LABEL = 10 * 4 * 7
SHUFFLE_SEED = None

LABEL_ORDER = []
for lb in LABELS_FOR_ORDER:
    LABEL_ORDER.extend([lb] * int(ORDER_REPEATS_PER_LABEL))
random.Random(SHUFFLE_SEED).shuffle(LABEL_ORDER)

TARGET_HOURS_PER_LEVEL = 6 * 4 * 7
SEGMENT_SECONDS = 300.0
WINDOW_SECONDS = 256.0
STEP_SECONDS = 256.0
RANDOM_SEED = None

PARALLEL_FEATURES = True
N_JOBS = max(1, os.cpu_count() - 1)

# ---- Helpers ----

def _build_balanced_plan(order, target_seconds_per_label, segment_seconds):
    remaining = {label: float(target_seconds_per_label) for label in set(order)}
    plan = []
    idx = 0
    while any(v > 0 for v in remaining.values()):
        label = order[idx % len(order)]
        idx += 1
        if remaining[label] <= 0:
            continue
        seg = min(float(segment_seconds), remaining[label])
        plan.append((label, seg))
        remaining[label] -= seg
    return plan


def _pick_segments_for_label(concat, label, target_seconds, rng):
    concat._load_label_files(label)
    files = list(concat.label_files.get(label, []))
    if not files:
        raise ValueError(f"No files available for label={label}")

    segments = []
    total = 0.0
    safety = 0
    while total < target_seconds:
        safety += 1
        if safety > 20000:
            raise RuntimeError("Safety stop: too many iterations while building segment.")

        filename = rng.choice(files)
        full_path = concat._get_full_path(label, filename)
        df = concat.data_cache[full_path]
        duration = concat._get_duration_from_dataframe(df)
        if duration <= 0:
            continue

        segments.append((label, filename))
        total += duration

    return segments


def _compute_feature(window_item, source_name):
    window_id, t_start, t_end, window_df = window_item
    return extract_features_from_window(
        g=window_df,
        window_id=window_id,
        source_file=source_name,
        window_start=t_start,
        window_end=t_end,
    )


def _append_and_extract_windows(
    buffer_df,
    new_df,
    window_start,
    window_seconds,
    step_seconds,
    source_name,
    window_id_start,
    parallel_features,
    n_jobs,
    ):
    if buffer_df is None:
        buffer_df = new_df
    else:
        buffer_df = pd.concat([buffer_df, new_df], ignore_index=True)

    time_vals = buffer_df["Time"].to_numpy()
    if len(time_vals) == 0:
        return buffer_df, window_start, window_id_start, []
    if window_start is None:
        window_start = float(time_vals[0])

    window_items = []
    while time_vals[-1] >= window_start + window_seconds - 1e-6:
        idx_start = int(np.searchsorted(time_vals, window_start, side="left"))
        idx_end = int(np.searchsorted(time_vals, window_start + window_seconds, side="left"))
        if idx_end > idx_start:
            window_df = buffer_df.iloc[idx_start:idx_end]
            window_items.append((window_id_start, window_start, window_start + window_seconds, window_df))
        window_id_start += 1
        window_start += step_seconds

    if not window_items:
        return buffer_df, window_start, window_id_start, []
    if parallel_features and len(window_items) > 1:
        feats = Parallel(n_jobs=n_jobs, prefer="threads")(
            delayed(_compute_feature)(item, source_name) for item in window_items
        )
        features = [f for f in feats if f is not None]
    else:
        features = []
        for item in window_items:
            feat = _compute_feature(item, source_name)
            if feat is not None:
                features.append(feat)

    keep_idx = int(np.searchsorted(time_vals, window_start, side="left"))
    if keep_idx > 0:
        buffer_df = buffer_df.iloc[keep_idx:].reset_index(drop=True)

    return buffer_df, window_start, window_id_start, features


def _extract_features_streaming(concat, plan, window_seconds, step_seconds, source_name):
    features_rows = []
    offset = 0.0
    window_id_global = 0
    window_start = None
    buffer_df = None

    for label, seg_seconds in plan:
        segments = _pick_segments_for_label(concat, label, seg_seconds, rng)
        seg_df, majority_label, _ = concat._build_duration_segment(segments, seg_seconds)
        if majority_label != label:
            raise ValueError(f"Majority label mismatch: expected {label}, got {majority_label}")
        if "label" not in seg_df.columns:
            seg_df["label"] = label
        seg_df = ECGAdvancedConcatenator._offset_time(seg_df, offset)
        offset += seg_seconds

        buffer_df, window_start, window_id_global, new_feats = _append_and_extract_windows(
            buffer_df,
            seg_df,
            window_start,
            window_seconds,
            step_seconds,
            source_name,
            window_id_global,
            PARALLEL_FEATURES,
            N_JOBS,
        )
        if new_feats:
            features_rows.extend(new_feats)

    return features_rows


# ---- Streaming feature extraction ----
rng = random.Random(RANDOM_SEED)
run_dir = Path(OUTPUT_ROOT_DIR) / str(DATASET_ID)
run_dir.mkdir(parents=True, exist_ok=True)

concat = ECGAdvancedConcatenator(
    csv_label_file=None,
    data_dir=RAW_BASE_DIR,
    labels=sorted(set(LABEL_ORDER)),
)

plan = _build_balanced_plan(
    LABEL_ORDER,
    target_seconds_per_label=TARGET_HOURS_PER_LEVEL * 3600.0,
    segment_seconds=SEGMENT_SECONDS,
)

source_name = f"balanced_interleaved_id{DATASET_ID}.csv"
features_rows = _extract_features_streaming(
    concat,
    plan,
    window_seconds=WINDOW_SECONDS,
    step_seconds=STEP_SECONDS,
    source_name=source_name,
)

if not features_rows:
    raise ValueError("No features extracted. Check window settings.")

features_df = pd.DataFrame(features_rows)
features_df["label"] = features_df["label"].astype(int)

features_dir = Path(FEATURES_DIR)
features_dir.mkdir(parents=True, exist_ok=True)

features_out = features_dir / f"features_windowed_balanced_{int(WINDOW_SECONDS)}s_id{DATASET_ID}.csv"
features_df.to_csv(features_out, index=False)

label_counts = features_df["label"].value_counts().sort_index()
stratify_col = features_df["label"] if label_counts.min() >= 2 else None

print(f"Saved features: {features_out}")
print("Label distribution:")
print(label_counts)

Saved features: c:\Users\buck\Napplee\StressClassification\data\features\7\features_windowed_balanced_256s_id7.csv
Label distribution:
label
0    2386
1    2332
2    2385
3    2347
Name: count, dtype: int64
